# 02 — Full-state flow-map baseline

Questo notebook verifica se la flow map Hay da 1 ms contiene un segnale apprendibile e componibile in rollout brevi. Non implementa HayFlow-Hines, latenti persistenti, riduzione morfologica, Mamba o S4.

Il solo dataset accettato è `hayflow_diagnostic_dataset_v1.zip`, schema `1.0.1`. Il notebook rifiuta esplicitamente `1.0.0`, hash non validi e teacher differenti.

## 1. Checkout riproducibile

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

ELM_REPOSITORY = "https://github.com/Zagred47/giada.git"
ELM_REF = os.environ.get("HAYFLOW_ELM_REF", "main")
NOTEBOOK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd().resolve()
WORKSPACE = NOTEBOOK_ROOT / "hayflow_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

elm_override = os.environ.get("HAYFLOW_ELM_REPO")
mounted = [Path(elm_override).expanduser()] if elm_override else []
mounted.extend([Path.cwd(), *Path.cwd().parents])
mounted_elm = next((p.resolve() for p in mounted if (p / "src" / "hayflow_model").is_dir()), None)
if mounted_elm is not None:
    ELM_REPO = mounted_elm
else:
    ELM_REPO = WORKSPACE / "elmneuron"
    if not (ELM_REPO / ".git").is_dir():
        run(["git", "clone", ELM_REPOSITORY, ELM_REPO])
    run(["git", "fetch", "origin", ELM_REF], cwd=ELM_REPO)
    run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ELM_REPO)
os.environ["HAYFLOW_ELM_REPO"] = str(ELM_REPO)
sys.path.insert(0, str(ELM_REPO))
for module_name in tuple(sys.modules):
    if module_name == "src.hayflow_data" or module_name.startswith("src.hayflow_data.") or module_name == "src.hayflow_model" or module_name.startswith("src.hayflow_model.") or module_name == "src.hayflow_eval" or module_name.startswith("src.hayflow_eval."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()
print("Owned repository:", ELM_REPO)

## 2. Dipendenze e GPU

Il profilo scientifico completo richiede una GPU Kaggle. Il profilo `smoke` serve soltanto a verificare il cablaggio e non può produrre una decisione scientifica.

In [ ]:
run([sys.executable, "-m", "pip", "install", "--quiet", "h5py", "pyarrow", "pyyaml", "pandas", "matplotlib"])
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Dataset canonico `v1.0.1`

Il controllo legge tutti i manifest, il report finale e i 173 SHA-256. RNG e release non vengono trasformati in falsi target continui.

In [ ]:
dataset_override = os.environ.get("HAYFLOW_DIAGNOSTIC_DATASET")
dataset_candidates = [Path(dataset_override).expanduser()] if dataset_override else []
if Path("/kaggle/input").is_dir():
    dataset_candidates.extend(Path("/kaggle/input").rglob("hayflow_diagnostic_dataset_v1.zip"))
    dataset_candidates.extend(path.parent for path in Path("/kaggle/input").rglob("dataset_manifest.json"))
DATASET_SOURCE = next((path.resolve() for path in dataset_candidates if path.exists()), None)
assert DATASET_SOURCE is not None, (
    "Dataset v1.0.1 non trovato. Aggiungi hayflow_diagnostic_dataset_v1.zip "
    "agli input Kaggle oppure imposta HAYFLOW_DIAGNOSTIC_DATASET."
)
print("Dataset source:", DATASET_SOURCE)

In [ ]:
from src.hayflow_data import prepare_flowmap_bundle

BUNDLE_CACHE = WORKSPACE / "diagnostic_dataset_v1_0_1"
bundle = prepare_flowmap_bundle(
    DATASET_SOURCE,
    cache_dir=BUNDLE_CACHE,
    verify_hashes=True,
)
display({
    "schema_version": bundle.schema_version,
    "teacher_commit": bundle.manifest["teacher_commit"],
    "transition_count": bundle.manifest["transition_count"],
    "event_count": bundle.manifest["event_count"],
    "artifact_validation": bundle.artifact_validation,
})
assert bundle.schema_version == "1.0.1"

## 4. Contratto dell’esperimento

B0 è persistence; B1 è ridge duale; B2 è un MLP globale controllato; B3 usa token di variabile associati ai segmenti, pesi condivisi, contesto parent/children, regione e morfologia. Le ablation confrontano voltage-only, stato completo, U1/U2 e privileged loss.

In [ ]:
import yaml
from src.hayflow_model import FlowmapExperimentConfig, FullStateFlowmapExperiment

CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "full_state_flowmap_baseline.yml"
config_document = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
raw = dict(config_document["experiment"])
raw["initialization_seeds"] = tuple(raw["initialization_seeds"])
raw["rollout_horizons_ms"] = tuple(raw["rollout_horizons_ms"])
raw["profile"] = os.environ.get("HAYFLOW_02_PROFILE", raw["profile"])
experiment_config = FlowmapExperimentConfig(**raw)
if experiment_config.profile == "diagnostic_full" and not torch.cuda.is_available():
    raise RuntimeError("Abilita un acceleratore GPU Kaggle prima del profilo diagnostic_full.")
OUTPUT_DIR = ELM_REPO / "artifacts" / "full_state_flowmap_baseline"
experiment = FullStateFlowmapExperiment(bundle, OUTPUT_DIR, experiment_config)
display({"profile": experiment_config.profile, "output_dir": str(OUTPUT_DIR), "config": raw})

## 5. Normalizzazione train-only e preflight

Il preflight costruisce ogni famiglia di modello, esegue un forward finito e verifica dimensioni, privileged heads e anti-leakage prima di iniziare il training lungo.

In [ ]:
preparation = experiment.prepare()
preflight = experiment.preflight()
display({
    "preparation": preparation,
    "preflight_valid": preflight["valid"],
    "device": preflight["device"],
    "model_parameter_counts": [row["parameter_count"] for row in preflight["models"]],
    "release_contract": preflight["release_contract"],
})
assert preflight["valid"]
assert not preflight["release_contract"]["rng_used_as_regression_target"]

## 6. B0–B3, ablation e rollout

Questa è la cella lunga. Mostra epoca, percentuale ed ETA; salva `last.pt` e `best.pt` per ogni run. Se la sessione si interrompe, rieseguendo il notebook sulla stessa directory il training riparte dal checkpoint compatibile. Con tre seed e tutte le ablation può richiedere diverse ore: è un esperimento, non una cella interattiva breve.

In [ ]:
final_report = experiment.run()
display({
    "best_baseline": final_report["best_baseline_by_validation_voltage_rmse"],
    "decision": final_report["decision"],
    "rng_and_release_contract": final_report["rng_and_release_contract"],
    "limitations": final_report["limitations"],
})

## 7. Controllo degli output

Il profilo completo deve produrre metriche one-step, rollout, eventi, ablation, esempi, figure, checkpoint e decisione finale. `GO` non significa che HayFlow sia risolto: autorizza soltanto il prototipo strutturato successivo.

In [ ]:
required_files = [
    "experiment_manifest.json", "normalization_schema.json",
    "model_configs.json", "model_registry.json",
    "one_step_metrics.parquet", "rollout_metrics.parquet",
    "event_metrics.parquet", "ablation_metrics.parquet",
    "training_history.json", "branching_report.json", "final_report.json",
]
missing = [name for name in required_files if not (OUTPUT_DIR / name).is_file()]
assert not missing, missing
assert any((OUTPUT_DIR / "checkpoints").rglob("best.pt"))
assert any((OUTPUT_DIR / "prediction_examples").glob("*.npz"))
assert any((OUTPUT_DIR / "figures").glob("*.png"))
print("Output completo:", OUTPUT_DIR)

## 8. Download dell’artefatto

La cella usa il download Blob che funziona in Kaggle: crea lo ZIP, lo trasferisce in Base64 al browser e avvia un vero `<a download>` temporaneo.

In [ ]:
import base64
from shutil import make_archive
from IPython.display import Javascript, display

archive_base = NOTEBOOK_ROOT / "hayflow_full_state_flowmap_baseline"
zip_path = Path(make_archive(
    str(archive_base),
    "zip",
    root_dir=OUTPUT_DIR.parent,
    base_dir=OUTPUT_DIR.name,
))
encoded = base64.b64encode(zip_path.read_bytes()).decode("ascii")
filename = zip_path.name
display(Javascript(f"""
(() => {{
  const binary = atob('{encoded}');
  const bytes = new Uint8Array(binary.length);
  for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
  const blob = new Blob([bytes], {{type: 'application/zip'}});
  const url = URL.createObjectURL(blob);
  const anchor = document.createElement('a');
  anchor.href = url;
  anchor.download = '{filename}';
  document.body.appendChild(anchor);
  anchor.click();
  anchor.remove();
  setTimeout(() => URL.revokeObjectURL(url), 60000);
}})();
"""))
print("Download avviato:", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")